In [3]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam


In [ ]:
df = pd.read_csv(r'C:\Users\YASHVIR\OneDrive\Attachments\PROJECTS\NLP\imdb\dataset\cleaned_data.csv')
print(f"Loaded {len(df)} cleaned rows")
 
X = df['clean_review'].astype(str)
y = df['label']


Loaded 49582 cleaned rows


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Bidirectional, Dense, Dropout, SpatialDropout1D)

In [6]:
X_train, X_test,y_train,y_test = train_test_split(X,y, test_size=0.25, random_state=42, stratify=y)


In [7]:

MAX_WORDS = 15000
MAX_LEN = 200
 
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
 
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
 
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")
 

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
class_weight_dict = dict(enumerate(class_weights))


In [ ]:
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])
 
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
 
model.summary()
 

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
    ModelCheckpoint("best_lstm_model.keras", monitor="val_loss", save_best_only=True),
]
 

In [13]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=15,                 # EarlyStopping will cut this short if it plateaus
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)
 

Epoch 1/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 80s 165ms/step - accuracy: 0.8028 - loss: 0.4285 - val_accuracy: 0.8705 - val_loss: 0.3078 - learning_rate: 0.0010
Epoch 2/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 74s 159ms/step - accuracy: 0.9113 - loss: 0.2388 - val_accuracy: 0.8813 - val_loss: 0.3019 - learning_rate: 0.0010
Epoch 3/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 71s 153ms/step - accuracy: 0.9375 - loss: 0.1790 - val_accuracy: 0.8774 - val_loss: 0.3450 - learning_rate: 0.0010
Epoch 4/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 70s 150ms/step - accuracy: 0.9658 - loss: 0.1046 - val_accuracy: 0.8738 - val_loss: 0.3990 - learning_rate: 5.0000e-04
Epoch 5/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 68s 145ms/step - accuracy: 0.9796 - loss: 0.0664 - val_accuracy: 0.8736 - val_loss: 0.5279 - learning_rate: 2.5000e-04


In [14]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()
 
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))
 

388/388 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step

Accuracy: 0.8821393998063891
              precision    recall  f1-score   support

    Negative       0.88      0.89      0.88      6175
    Positive       0.89      0.88      0.88      6221

    accuracy                           0.88     12396
   macro avg       0.88      0.88      0.88     12396
weighted avg       0.88      0.88      0.88     12396



In [15]:
model.save('model.keras')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
 
print("\nSaved model.keras and tokenizer.pkl")


Saved model.keras and tokenizer.pkl
